In [1]:
import pandas as pd
import numpy as np
from statsmodels.stats.proportion import proportion_confint
import statsmodels.api as sm
import statsmodels.formula.api as smf

In [2]:
events = pd.read_parquet(f"../data/events.parquet")
races = pd.read_parquet(f"../data/races.parquet")
runners = pd.read_parquet(f"../data/runners.parquet")

In [3]:
fbase = "../results/tables"

# Pre-prep data

## Logit

In [4]:
runners["logit_prob"] = np.log(runners.implied_odds/(1-runners.implied_odds))

## Normalised Probabilities

In [5]:
book_pct = runners.groupby("race_id").implied_odds.sum()
runners["normalised_odds"] = runners.implied_odds/runners.race_id.map(book_pct)

## Test Statistics

In [6]:
runners["returns"] = runners.bsp * runners.win_lose
runners["brier"]   = np.square(runners.normalised_odds - runners.win_lose)
runners["log_loss"] = -(
    np.log(runners.normalised_odds)*runners.win_lose +
    np.log(1-runners.normalised_odds)*(1-runners.win_lose)
)

/home/mabel/.pyenv/versions/3.14.2/lib/python3.14/site-packages/pandas/core/arraylike.py:402: RuntimeWarning: divide by zero encountered in log
  result = getattr(ufunc, method)(*inputs, **kwargs)


## Bsp Binning Range

In [7]:
ticks = np.concat((
    np.arange(1, 2, 0.01),
    np.arange(2,3,0.02),
    np.arange(3,4,0.05),
    np.arange(4,6,0.1),
    np.arange(6,10,0.2),
    np.arange(10,20,0.5),
    np.arange(20,30,1),
    np.arange(30,50,2),
    np.arange(50,100,5),
    np.arange(100,1000,10),
    [1001]
))
tick_range=[0,2,3,4,6,10,20,30,50,100,1001]
tick_size=[0.01, 0.02, 0.05, 0.1, 0.2, 0.5, 1, 2, 5, 10]
#ticks=1000
t = {
    1: 0.01,
    2: 0.02,
    3: 0.05,
    4: 0.1,
    6: 0.2,
    10: 0.5,
    20: 1,
    30: 2,
    50: 5,
    100: 10
}

# Favourite-Longshot Bias

## All

In [8]:
g = runners.groupby(pd.cut(runners.bsp, bins=ticks), observed=False)

In [9]:
lower, upper = proportion_confint(
    count=g.win_lose.sum(),
    nobs=g.size(),
    alpha = 0.05,
    method="wilson"
)

/home/mabel/.pyenv/versions/3.14.2/lib/python3.14/site-packages/statsmodels/stats/proportion.py:202: RuntimeWarning: invalid value encountered in divide
  q_ = count_a / nobs_a
/home/mabel/.pyenv/versions/3.14.2/lib/python3.14/site-packages/statsmodels/stats/proportion.py:291: RuntimeWarning: divide by zero encountered in divide
  denom = 1 + crit2 / nobs_a
/home/mabel/.pyenv/versions/3.14.2/lib/python3.14/site-packages/statsmodels/stats/proportion.py:292: RuntimeWarning: divide by zero encountered in divide
  center = (q_ + crit2 / (2 * nobs_a)) / denom
/home/mabel/.pyenv/versions/3.14.2/lib/python3.14/site-packages/statsmodels/stats/proportion.py:293: RuntimeWarning: divide by zero encountered in divide
  dist = crit * np.sqrt(q_ * (1.0 - q_) / nobs_a + crit2 / (4.0 * nobs_a**2))


In [10]:
all_df = pd.DataFrame({"mean": g.win_lose.mean(), "count": g.size(), "volume": g.pptradedvol.sum(), "confidence-l": lower, "confidence-u": upper})

In [11]:
all_df["tick_size"] = np.array(tick_size)[pd.cut(all_df.index.categories.right, bins=tick_range, labels=False)]
#df = df.rolling(5).mean()
#all_df = all_df.rolling(5, min_periods=1).mean()
all_df["l"] = all_df.index.categories.left
all_df["r"] = all_df.index.categories.right
all_df.index=1/all_df.l

In [12]:
all_df.to_csv(f"{fbase}/all-fl.csv")

In [13]:
X = sm.add_constant(runners.logit_prob)
Y = runners.win_lose

model = sm.Logit(Y, X).fit()

model.summary()

Optimization terminated successfully.
         Current function value: 0.291088
         Iterations 7


<class 'statsmodels.iolib.summary.Summary'>
"""
                           Logit Regression Results                           
==============================================================================
Dep. Variable:               win_lose   No. Observations:              1594573
Model:                          Logit   Df Residuals:                  1594571
Method:                           MLE   Df Model:                            1
Date:                Sun, 18 Oct 2026   Pseudo R-squ.:                  0.1630
Time:                        13:32:23   Log-Likelihood:            -4.6416e+05
converged:                       True   LL-Null:                   -5.5455e+05
Covariance Type:            nonrobust   LLR p-value:                     0.000
==============================================================================
                 coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------
const         -0.0064      0.005     -1.180      0.238      -0.017       0.004
logit_prob     0.9967      0.003    360.286      0.000       0.991       1.002
==============================================================================
"""

In [14]:
model.conf_int()

,0,1
const,-0.016927,0.004203
logit_prob,0.991311,1.002156


In [15]:
model.params

const        -0.006362
logit_prob    0.996733
dtype: float64

## Groups

per race stats

In [16]:
rg = runners.groupby("race_id")

nraces = races.assign(
    ppvol=rg.pptradedvol.sum(),
    nrunners=rg.size(),
    overround=rg.implied_odds.sum()
)

add groups that are needed

In [17]:
df = runners.join(nraces, on="race_id").join(events, on="event_id")
df["hour"] = df.race_time.dt.hour
df["race_ppvol"] = df.race_id.map(pd.qcut(nraces.ppvol, q=10))
df["overround"] = df.race_id.map(pd.qcut(nraces.overround, q=10))

In [18]:
cats = {
    "race_no": ("race_no", [1,2,3,4,5,6,7,8]),
    "distance_f": ("distance_f", df.distance_f.value_counts().head(10).index),
    "hour": ("hour", np.arange(12,20,1)),
    "race_type": ("race_type", df.race_type.value_counts().head(10).index),
    "location": ("location", df.location.value_counts().head(10).index),
    "nrunners": ("nrunners", df.nrunners.value_counts().head(10).index),
    "race_ppvol": ("race_ppvol", pd.qcut(nraces.ppvol, q=10)),
    "overround": ("overround", pd.qcut(nraces.overround, q=10))
}

In [19]:
def regress(x):
    try:
        model = sm.Logit(x.win_lose, sm.add_constant(x.logit_prob)).fit(disp=False)
        ci = model.conf_int()
        ilower, slower = ci[0]
        iupper, supper = ci[1]
    except:
        return pd.Series({"slope": 0, "intercept": 0, "ilower": 0, "iupper": 0, "slower": 0, "supper": 0})

    df = pd.Series({"slope": model.params["logit_prob"], "intercept": model.params["const"], "ilower": ilower, "iupper": iupper, "slower": slower, "supper": supper})
    return df

In [20]:
for c in cats:
    dfs = dict()
    for a, v in [("mean", "win_lose"), ("count", "win_lose"), ("sum", "pptradedvol"), ("sum", "win_lose")]:
        ndf = df[df[c].isin(cats[c][1])]
        pt = ndf.pivot_table(
            index=pd.cut(ndf.bsp, bins=ticks),
            columns=cats[c][0],
            values=v,
            aggfunc=a,
            observed=False,
            dropna=False
        )
        pt = pt.reindex(sorted(pt.columns), axis=1)
        pt.columns = pt.columns.map(str).str.replace(",", ";")
        ts = np.array(tick_size)[pd.cut(pt.index.categories.right, bins=tick_range, labels=False)]
        pt.insert(0, 'tick_size', ts)
        #pt=pt.groupby("tick_size", as_index=False).rolling(5, min_periods=1).mean()
        #pt = pt.rolling(5).mean()
        l = pt.index.categories.left
        r = pt.index.categories.right
        
        pt.index=r
        dfs[(a,v)] = pt
        out = pt.rolling(5).mean() if a in ("count", "sum") else pt   # smooth plotted frequencies only
        out.to_csv(f"{fbase}/{c}-{v}-{a}.csv")

    # bias
    wr = dfs[("mean", "win_lose")]
    cal_error = wr.subtract(1/wr.index, axis=0)
    ce = pd.DataFrame({
        "Bias": cal_error.mean(),
        "Standard Deviation": cal_error.std(),
        "Count": dfs[("count", "win_lose")].sum()
    }).iloc[1:, :]
    se = ce["Standard Deviation"] / np.sqrt(ce["Count"])
    ce["ci_l"] = ce["Bias"] - 1.96 * se
    ce["ci_u"] = ce["Bias"] + 1.96 * se
    #print(ce)
    ce.to_csv(f"{fbase}/{c}-calib-error.csv")

    # regression
    # WARNINGS HERE DO NOT MATTER AS ALL CORRESPOND TO FILTERED DATA
    df[df[c].isin(cats[c][1])].groupby(c).apply(regress).sort_index().to_csv(f"{fbase}/{c}-regress.csv")

# Test Statistics

## All

In [21]:
all_brier = runners.groupby("race_id").brier.mean().mean()
all_ll = runners.groupby("race_id").log_loss.mean().mean()

In [22]:
print(f"Brier: {all_brier}")
print(f"Log-Loss: {all_ll}")

Brier: 0.09460758799746073
Log-Loss: 0.3156829995247215


In [23]:
b = runners.groupby("race_id").brier.mean()
r = []
for i in range(20):
    r += [b.sample(17000, replace=True, random_state=i).mean()]
r = np.array(r)
np.percentile(r, 2.5), np.percentile(r, 97.5)

(np.float64(0.09394094190258422), np.float64(0.09531687472899966))

In [24]:
def simulate_ci(series, group, n=1000, seed=42, quantiles=[0.025, 0.975]):
    groups = {
        name: s.to_numpy()
        for name, s in series.groupby(group)
    }

    rng = np.random.default_rng(seed)

    r = []
    for i in range(n):
        r+=[{
            name: rng.choice(values, size=len(values), replace=True).mean()
            for name, values in groups.items()
        }]

    return pd.DataFrame(r).quantile(quantiles).T

In [25]:
b = []
l = []
r = np.arange(0.02,1,0.02)
for x in r:
    b.append(np.square(x - runners.win_lose).groupby(runners.race_id).mean().mean())
    l.append(-(np.log(x)*runners.win_lose + np.log(1-x)*(1-runners.win_lose)).groupby(runners.race_id).mean().mean())

ndf = pd.DataFrame({"brier": 1-b/all_brier, "log_loss": 1-l/all_ll}, index=r)
ndf.to_csv(f"{fbase}/all_ss_random.csv")

## Groups

In [26]:
for c in cats:
    ndf = df[df[c].isin(cats[c][1])]
    bci = simulate_ci(ndf.groupby(["race_id", c]).brier.mean(), c)
    lci = simulate_ci(ndf.groupby(["race_id", c]).log_loss.mean(), c)
    
    b = ndf.groupby(["race_id", cats[c][0]]).brier.mean().groupby(level=1).mean()
    l = ndf.groupby(["race_id", cats[c][0]]).log_loss.mean().groupby(level=1).mean()

    f = pd.DataFrame({"brier": 1-b/all_brier,
                      "log-loss": 1-l/all_ll,
                      "count": ndf.groupby(["race_id", cats[c][0]]).size().groupby(level=1).size(),
                      "volume": ndf.groupby(cats[c][0]).pptradedvol.sum()})
    f = f.join(1-bci/all_brier, rsuffix="b")
    f = f.join(1-lci/all_ll, rsuffix="l")
    #print(f)
    #f = f.loc[cats[c][1], :]
    f.drop_duplicates(inplace=True)
    f = f.sort_index()
    f.to_csv(f"{fbase}/{c}-acc.csv")

# Trading

In [27]:
all_favourites = df.sort_values("bsp").groupby("race_id").head(1)

In [28]:
def create_bets(slip, mode="abs"):
    bets=slip
    # generate timeline
    #sort = ["race_time"] if not group else [group, "race_time"]
    bets = bets.join(races[["race_time"]], on="race_id", how="inner").sort_values("race_time")
    # pnl
    if mode == "abs":
        bets["profit"] = (bets.bsp * bets.win_lose -1) * bets.liability
        #bets["cum_profit"] = bets.groupby(group).profit.cumsum()
    elif mode == "pct":
        bets["profit"] = bets.liability*(bets.bsp*bets.win_lose-1)+1
    
    return bets

In [29]:
def bet_on_favourite(group=None):
    favs = all_favourites
    cols = ["race_id", "win_lose", "bsp"]
    if group:
        cols += [group]
    slip = favs[cols]
    slip = slip.assign(liability=1)

    return create_bets(slip)

In [30]:
def hedge_on_favourite(group=None):
    favs = all_favourites
    cols = ["race_id", "win_lose", "bsp"]
    if group:
        cols += [group]
    slip = favs[cols]
    slip = slip.assign(liability=favs.implied_odds)

    return create_bets(slip)

In [31]:
def harville_odds(runners):
    p = runners.normalised_odds
    n=len(p)
    places = runners.place_n.iloc[0]

    if not (places == 2 or places == 3):
        p[:] = 0
        return p
    
    second = p * ((p / (1-p)).sum() - (p / (1-p)))

    place = p + second

    if places == 2:
        return place

    # broadcast over i, j, k
    p = p.to_numpy()
    pi = p[:, None, None]
    pj = p[None, :, None]
    pk = p[None, None, :]

    third = pj * pk / (1 - pj) * pi / (1 - pj - pk)

    # remove invalid cases
    i = np.arange(n)[:, None, None]
    j = np.arange(n)[None, :, None]
    k = np.arange(n)[None, None, :]

    third[(j == k) | (i == j) | (i == k)] = 0

    place += third.sum(axis=(1,2))

    return place

In [32]:
x = df.set_index(["race_id", df.index]).sort_index()

In [33]:
ho = x.groupby(level=0, as_index=False).apply(harville_odds)

In [34]:
x["harville_odds"] = ho

In [35]:
x = x.reset_index(level=0)

In [36]:
df = x

In [37]:
def bet_harville_edge(group=None):
    edge = df.harville_odds * df.place_bsp
    selections = df[edge > 1.25]
    
    p = selections.harville_odds
    q = 1-p
    b = selections.place_bsp-1

    cols = ["race_id", "place_win_lose", "place_bsp"]
    if group:
        cols += [group]
    
    slip = selections[cols]
    slip = slip.rename(columns = {"place_win_lose": "win_lose", "place_bsp": "bsp"})
    slip = slip.assign(liability=p-q/b)
    slip = slip[~slip.liability.isna()]
    slip.liability = slip.liability.clip(lower=0)
    slip = slip[slip.liability > 0]
    slip.liability *= 0.1

    return create_bets(slip, "pct")
    

In [38]:
def sharpe_ratio(ret):
    avg_return = ret.mean()
    std_dev = ret.std()

    return avg_return/std_dev

In [39]:
strategies = {
    "bet-fav": (bet_on_favourite, "sum"),
    "hedge-fav": (hedge_on_favourite, "sum"),
    "bet-harville": (bet_harville_edge, "prod")
}

## All

In [40]:
results = dict()
for strat, (f, agg) in strategies.items():
    bets = f()
    n = len(bets)

    p = []
    for i in range(1000):
        sample = bets.profit.sample(n, replace=True, random_state = i)
        p += [sample.mean()]
    results[strat] = p

results = pd.DataFrame(results)
results["bet-harville"]-=1

In [41]:
frame = results.quantile([0.025, 0.975]).T
frame["mean"] = results.mean()
frame["sharpe"] = sharpe_ratio(results)
frame.to_csv(f"{fbase}/all-strats.csv")

## Groups

In [42]:
for c in cats:
    print(c)
    results = dict()
    for strat, (f, agg) in strategies.items():
        print(" "+strat)
        bets = f(c)
        bets = bets[bets[c].isin(cats[c][1])]
        n = len(bets)
    
        p = [0]*1000
        for i in range(1000):
            sample = bets.groupby(c).sample(frac=0.1, replace=True, random_state = i).groupby(c).profit
            p[i] = sample.mean()
        
        k=pd.DataFrame(p)
        if strat=="bet-harville":
            k-=1
        frame = k.quantile([0.025, 0.975]).T
        frame["mean"] =k.mean()
        frame["sharpe"] = sharpe_ratio(k)
        frame.to_csv(f"{fbase}/{c}-{strat}.csv")

race_no
 bet-fav
 hedge-fav
 bet-harville
distance_f
 bet-fav
 hedge-fav
 bet-harville
hour
 bet-fav
 hedge-fav
 bet-harville
race_type
 bet-fav
 hedge-fav
 bet-harville
location
 bet-fav
 hedge-fav
 bet-harville
nrunners
 bet-fav
 hedge-fav
 bet-harville
race_ppvol
 bet-fav
 hedge-fav
 bet-harville
overround
 bet-fav
 hedge-fav
 bet-harville
